# Enrich Dataset

In [124]:
import pandas as pd
import re

In [125]:
emb_desc = pd.read_csv("/data/giobbi/embeddings/DESC_GPT.csv", sep="\t", index_col=0)
emb_desc = emb_desc[["Drug ID", "Drug Name", "Discription"]]


## Drug Count

In [126]:
from flashtext import KeywordProcessor

# Work on a copy
emb_desc_masked = emb_desc.copy()

# Ensure string types where needed
emb_desc_masked['Drug ID'] = emb_desc_masked['Drug ID'].astype(str)
emb_desc_masked['Drug Name'] = emb_desc_masked['Drug Name'].astype(str)
emb_desc_masked['Discription'] = emb_desc_masked['Discription'].astype(str)

# 1) Build a masker that replaces any Drug Name or Drug ID with "<DRUG>"
kp_mask = KeywordProcessor(case_sensitive=False)
all_terms = pd.concat([
    emb_desc_masked['Drug Name'].dropna(),
    emb_desc_masked['Drug ID'].dropna()
]).astype(str).drop_duplicates()
for term in all_terms:
    kp_mask.add_keyword(term, "<DRUG>")

# 2) Build a canonical mapper that maps both Drug Name and Drug ID -> canonical Drug ID
kp_id = KeywordProcessor(case_sensitive=False)
canonical_pairs = emb_desc_masked[['Drug ID', 'Drug Name']].dropna().drop_duplicates()
for _, row in canonical_pairs.iterrows():
    did = str(row['Drug ID'])
    dname = str(row['Drug Name'])
    # map ID -> ID and Name -> ID so synonyms collapse
    kp_id.add_keyword(did, did)
    kp_id.add_keyword(dname, did)

# Apply masking and counting
emb_desc_masked['Description_Masked'] = emb_desc_masked['Discription'].apply(
    lambda x: kp_mask.replace_keywords(x) if pd.notnull(x) else x
)
emb_desc_masked['drug_unique_count'] = emb_desc_masked['Discription'].apply(
    lambda x: len(set(kp_id.extract_keywords(x))) if pd.notnull(x) else 0
)
emb_desc_masked['drug_count'] = emb_desc_masked['Discription'].apply(
    lambda x: len(kp_id.extract_keywords(x)) if pd.notnull(x) else 0
)

emb_desc_masked.sort_values(by='drug_unique_count', ascending=False, inplace=True)
emb_desc_masked.reset_index(drop=True, inplace=True)

# Quick preview
emb_desc_masked[['Drug ID', 'Drug Name', 'drug_unique_count', 'drug_count']].head()

,Drug ID,Drug Name,drug_unique_count,drug_count
0,DB14732,Queuine,13,19
1,DB00736,Esomeprazole,9,13
2,DB01026,Ketoconazole,9,13
3,DB11755,Tetrahydrocannabivarin,9,10
4,DB00213,Pantoprazole,9,9


## Drug Class

In [127]:
# Expanded keyword dictionary
keyword_classes = {
    0: [  # Antidepressants
        "antidepressant", "tricyclic", "tca", "ssri", "snri", "maoi",
        "serotonin reuptake", "norepinephrine reuptake", "sertraline",
        "fluoxetine", "paroxetine", "citalopram", "escitalopram", "venlafaxine",
        "duloxetine", "mirtazapine", "bupropion"
    ],
    1: [  # Antifungals
        "antifungal", "azole", "ergosterol", "ketoconazole", "fluconazole",
        "itraconazole", "voriconazole", "posaconazole", "amphotericin",
        "echinocandin", "caspofungin", "micafungin", "anidulafungin"
    ],
    2: [  # Antihypertensives
        "antihypertensive", "hypertension", "blood pressure", "angiotensin",
        "arb", "ace inhibitor", "calcium channel blocker", "ccb", "beta-blocker",
        "losartan", "valsartan", "amlodipine", "lisinopril", "enalapril",
        "atenolol", "metoprolol", "propranolol"
    ],
    3: [  # Antibiotics
        "antibiotic", "antibacterial", "macrolide", "penicillin", "amoxicillin",
        "ampicillin", "cephalosporin", "ceftriaxone", "cephalexin",
        "fluoroquinolone", "ciprofloxacin", "levofloxacin", "tetracycline",
        "doxycycline", "minocycline", "clindamycin", "azithromycin",
        "erythromycin", "bacterial infection"
    ],
    4: [  # Antivirals
        "antiviral", "protease inhibitor", "reverse transcriptase inhibitor",
        "integrase inhibitor", "hiv", "hepatitis", "influenza", "acyclovir",
        "valacyclovir", "oseltamivir", "remdesivir", "sofosbuvir"
    ],
    5: [  # Anticancer
        "antineoplastic", "chemotherapy", "kinase inhibitor",
        "monoclonal antibody", "cytotoxic", "oncology", "immunotherapy",
        "platinum-based", "cisplatin", "carboplatin", "paclitaxel", "docetaxel",
        "bevacizumab", "trastuzumab", "rituximab"
    ],
    6: [  # Analgesics
        "analgesic", "pain", "painkiller", "opioid", "morphine", "oxycodone",
        "hydrocodone", "fentanyl", "codeine", "nsaid", "ibuprofen",
        "naproxen", "diclofenac", "acetaminophen", "paracetamol"
    ],
    7: [  # Antidiabetics
        "antidiabetic", "diabetes", "insulin", "glp-1", "glp1",
        "sglt2 inhibitor", "metformin", "sulfonylurea", "glipizide",
        "glyburide", "pioglitazone", "rosiglitazone", "dapagliflozin",
        "empagliflozin", "liraglutide", "semaglutide"
    ],
    8: [  # Gastrointestinal
        "gastrointestinal", "proton pump inhibitor", "ppi",
        "omeprazole", "lansoprazole", "pantoprazole", "rabeprazole",
        "esomeprazole", "h2 receptor antagonist", "ranitidine",
        "famotidine", "cimetidine", "gerd", "acid reflux", "ulcer"
    ],
    9: [  # Corticosteroids / Anti-inflammatory
        "corticosteroid", "glucocorticoid", "steroid", 
        "prednisone", "prednisolone", "methylprednisolone", 
        "dexamethasone", "hydrocortisone", "betamethasone"
    ],
    10: [  # Hormonal / Endocrine therapy
        "estrogen", "estradiol", "estriol", "estrone", 
        "progesterone", "progestin", "contraceptive", "birth control",
        "hormone replacement", "hrt", "tamoxifen", "raloxifene",
        "clomiphene", "letrozole", "aromatase inhibitor"
    ],
    11: [  # Sedatives / Hypnotics / Anxiolytics
        "benzodiazepine", "benzo", "diazepam", "valium", 
        "lorazepam", "ativan", "alprazolam", "xanax", "clonazepam",
        "temazepam", "zolpidem", "z-drug", "eszopiclone", "lunesta",
        "zaleplon", "sleep", "insomnia", "hypnotic", "sedative",
        "anxiolytic", "buspirone", "ramelteon", "melatonin"
    ],
    12: [  # Respiratory (Asthma / COPD)
        "asthma", "copd", "bronchodilator", "inhaler", "inhaled corticosteroid",
        "beta-agonist", "saba", "laba", "albuterol", "salbutamol", "ventolin",
        "formoterol", "salmeterol", "terbutaline",
        "fluticasone", "budesonide", "beclomethasone", "mometasone",
        "tiotropium", "ipratropium", "anticholinergic inhaler",
        "montelukast", "leukotriene antagonist"
    ],
    13: [  # Contrast Agents (Radiology / Imaging)
        "contrast agent", "radiopaque", "iodinated contrast", "iodine contrast",
        "gadolinium", "mri contrast", "ct contrast", "iohexol", "iopamidol",
        "iopromide", "gadobutrol", "gadoterate", "gadodiamide", "ultrasound contrast",
        "microbubble"
    ],
    14: [  # Antipsychotics
        "antipsychotic", "neuroleptic", "schizophrenia", "bipolar disorder",
        "haloperidol", "chlorpromazine", "fluphenazine",  # typical
        "risperidone", "olanzapine", "quetiapine", "clozapine",
        "aripiprazole", "ziprasidone", "paliperidone", "lurasidone", "asenapine",  # atypical
        "dopamine antagonist", "serotonin dopamine antagonist"
    ],
    15: [  # Antimalarials
        "antimalarial", "malaria", "plasmodium", "chloroquine", "hydroxychloroquine",
        "mefloquine", "quinine", "artemisinin", "artemether", "lumefantrine",
        "atovaquone", "proguanil", "primaquine", "tafenoquine"
    ],
    16: [  # Antiparasitics (non-malarial)
        "antiparasitic", "helminth", "protozoa", "ivermectin", "albendazole",
        "mebendazole", "praziquantel", "metronidazole", "tinidazole"
    ],
    17: [  # Antiepileptics / Anticonvulsants
        "antiepileptic", "anticonvulsant", "seizure", "epilepsy",
        "valproate", "valproic acid", "carbamazepine", "phenytoin",
        "lamotrigine", "levetiracetam", "topiramate", "gabapentin", "pregabalin"
    ],
    18: [  # Immunosuppressants
        "immunosuppressant", "transplant", "cyclosporine", "tacrolimus",
        "sirolimus", "everolimus", "mycophenolate", "azathioprine"
    ],
    19: [  # Anticoagulants / Antiplatelets
        "anticoagulant", "blood thinner", "warfarin", "heparin", "enoxaparin",
        "apixaban", "rivaroxaban", "dabigatran", "clopidogrel", "ticagrelor",
        "aspirin"
    ],
    20: [  # Lipid-lowering agents
        "lipid-lowering", "cholesterol", "statin", "atorvastatin", "simvastatin",
        "rosuvastatin", "pravastatin", "lovastatin", "ezetimibe",
        "pcsk9 inhibitor", "alirocumab", "evolocumab"
    ],
    21: [  # Thyroid drugs
        "thyroid", "hypothyroid", "hyperthyroid", "levothyroxine", "liothyronine",
        "methimazole", "propylthiouracil"
    ],
    22: [  # Vitamins & Supplements
        "vitamin", "supplement", "vitamin d", "vitamin b12", "folic acid",
        "iron", "calcium", "magnesium", "zinc", "multivitamin"
    ],
    23: [  # Unknown / None
    ],
    24: [  # Antiallergics / Antihistamines
        "antiallergic", "antihistamine", "loratadine", "cetirizine",
        "fexofenadine", "diphenhydramine", "chlorpheniramine", 
        "desloratadine", "allergy"
    ],
    25: [  # Anesthetics
        "anesthetic", "anesthesia", "lidocaine", "bupivacaine", 
        "propofol", "ketamine", "sevoflurane", "desflurane", 
        "general anesthesia", "local anesthesia", "spinal anesthesia"
    ],
    26: [  # Headache / Migraine
        "migraine", "headache", "triptan", "sumatriptan", "rizatriptan",
        "eletriptan", "naratriptan", "almotriptan", "frovatriptan",
        "acetaminophen", "paracetamol", "ibuprofen", "naproxen"
    ],
    27: [  # Bladder / Urinary
        "bladder", "urinary", "overactive bladder", "oxybutynin", 
        "tolterodine", "solifenacin", "darifenacin", "mirabegron",
        "tamsulosin", "benign prostatic hyperplasia", "bph"
    ],
    28: [  # Cancer / Leukemia
        "leukemia", "lymphoma", "multiple myeloma", "acute lymphoblastic",
        "acute myeloid", "chronic myeloid", "imatinib", "dasatinib",
        "nilotinib", "rituximab", "blinatumomab", "chemotherapy",
        "oncology", "cytotoxic"
    ],
}

def assign_class(description):
    if pd.isnull(description):
        return 23  # Unknown / None

    desc = description.lower()
    class_counts = {i: 0 for i in keyword_classes}

    for class_idx, keywords in keyword_classes.items():
        for kw in keywords:
            if kw == "azole":  # special case: match suffix
                if re.search(r"\w+azole\b", desc):
                    class_counts[class_idx] += 1
            elif kw in desc:
                class_counts[class_idx] += 1

    # Choose the class with the highest count
    best_class = max(class_counts, key=class_counts.get)
    if class_counts[best_class] == 0:
        return 23 # No matches
    return best_class

# Apply classification
emb_desc_masked['pharma_class'] = emb_desc_masked['Discription'].apply(assign_class)

# Check distribution
#print(emb_desc_masked['pharma_class'].value_counts())
emb_desc_masked

,Drug ID,Drug Name,Discription,Description_Masked,drug_unique_count,drug_count,pharma_class
0,DB14732,Queuine,Queuine is a derivative of [7-Deazaguanine]. B...,<DRUG> is a derivative of [7-Deazaguanine]. Ba...,13,19,23
1,DB00736,Esomeprazole,"Esomeprazole, sold under the brand name Nexium...","<DRUG>, sold under the brand name Nexium, is a...",9,13,8
2,DB01026,Ketoconazole,Ketoconazole is an imidazole antifungal agent ...,<DRUG> is an imidazole antifungal agent used i...,9,13,1
3,DB11755,Tetrahydrocannabivarin,Tetrahydrocannabivarin (THCV) is a propyl anal...,<DRUG> (THCV) is a propyl analogue of tetrahyd...,9,10,23
4,DB00213,Pantoprazole,Pantoprazole is a first-generation proton pump...,<DRUG> is a first-generation proton pump inhib...,9,9,8
...,...,...,...,...,...,...,...
8718,DB14728,Mebenil,An obsolete fungicide used to control various ...,An obsolete fungicide used to control various ...,0,0,23
8719,DB14734,Cannabigerol,A natural product found in Cannabis sativa and...,A natural product found in Cannabis sativa and...,0,0,23
8720,DB14735,Cannabichromene,A natural product found in Cannabis sativa and...,A natural product found in Cannabis sativa and...,0,0,23
8721,DB14736,Cannabivarin,A natural product found in Cannabis sativa.;;;;,A natural product found in Cannabis sativa.;;;;,0,0,23


## Drug Class ATC

In [128]:
atc_path = "./input/atc_tree_full.csv"
atc_df = pd.read_csv(atc_path, sep=",")

In [129]:
atc_df.head()


,childID,parentID,childType,childTerm,parentTerm
0,D11AX06,D11AX,5,Mequinol,4: OTHER DERMATOLOGICAL PREPARATIONS IN ATC
1,J05AC02,J05AC,5,Rimantadine,"4: Cyclic amines, direct acting antivirals"
2,R02AB01,R02AB,5,Neomycin,4: Antibiotic throat preparations
3,D10AX02,D10AX,5,Resorcinol,4: Other anti-acne preparations for topical us...
4,D10AF01,D10AF,5,Clindamycin,4: Antiinfectives for treatment of acne


In [130]:
# Build ATC hierarchy mapping
def get_level_1_parent(child_id, atc_df_dict):
    """Recursively traverse up the ATC hierarchy to find the level 1 (root) term"""
    current_id = child_id
    visited = set()
    
    # Traverse up the hierarchy
    while current_id in atc_df_dict and current_id not in visited:
        visited.add(current_id)
        row = atc_df_dict[current_id]
        
        # Check if this is a level 1 (single letter code like 'A', 'D', 'J', etc.)
        if len(current_id) == 1:
            return row['childTerm']
        
        # Move to parent
        parent_id = row['parentID']
        if parent_id and parent_id in atc_df_dict:
            current_id = parent_id
        else:
            break
    
    return None

# Create efficient lookup dict: childID -> row data
atc_lookup = {}
for _, row in atc_df.iterrows():
    atc_lookup[row['childID']] = {
        'parentID': row['parentID'],
        'childTerm': row['childTerm'],
        'childType': row['childType']
    }

# Add level 1 ATC parent for each drug
atc_df['atc_class_lvl_1'] = atc_df['childID'].apply(lambda x: get_level_1_parent(x, atc_lookup))

print("Sample of enriched ATC data:")
print(atc_df[['childID', 'childTerm', 'childType', 'atc_class_lvl_1']].head(10))
print(f"\nTotal drugs with level 1 mapping: {atc_df['atc_class_lvl_1'].notna().sum()}")
print(f"\nLevel 1 categories found:")
print(atc_df['atc_class_lvl_1'].value_counts())

Sample of enriched ATC data:
   childID     childTerm  childType                           atc_class_lvl_1
0  D11AX06      Mequinol          5                        1: DERMATOLOGICALS
1  J05AC02   Rimantadine          5        1: ANTIINFECTIVES FOR SYSTEMIC USE
2  R02AB01      Neomycin          5               1: RESPIRATORY SYSTEM DRUGS
3  D10AX02    Resorcinol          5                        1: DERMATOLOGICALS
4  D10AF01   Clindamycin          5                        1: DERMATOLOGICALS
5  C01BC09    Ethacizine          5            1: CARDIOVASCULAR SYSTEM DRUGS
6  N04AA04  Procyclidine          5                   1: NERVOUS SYSTEM DRUGS
7  N05CA09   Vinbarbital          5                   1: NERVOUS SYSTEM DRUGS
8  A01AB18  Clotrimazole          5  1: ALIMENTARY TRACT AND METABOLISM DRUGS
9  R02AA14  Oxyquinoline          5               1: RESPIRATORY SYSTEM DRUGS

Total drugs with level 1 mapping: 6853

Level 1 categories found:
atc_class_lvl_1
1: ALIMENTARY TRACT AND METABO

In [131]:
df_cresc = pd.read_csv("/data/giobbi/GRAPH/drugbank_crescenddi_graph_wo_contradiction.csv", sep="\t")

In [132]:
drug_ids = set(df_cresc['Drug1']) | set(df_cresc['Drug2'])
emb_desc_filtered = emb_desc_masked[emb_desc_masked['Drug ID'].isin(drug_ids)]

In [133]:
orig_names = set(emb_desc_filtered['Drug Name'].apply(lambda x: x.lower()))
atc_names = set(atc_df['childTerm'].apply(lambda x: x.lower()))

In [134]:
print("unmatched drugs:", len(orig_names - atc_names))

unmatched drugs: 239


In [135]:
from rapidfuzz import process, fuzz

# Create a proper mapping from lowercase to original childTerm
atc_lowercase_to_original = {}
for _, row in atc_df.iterrows():
    atc_lowercase_to_original[row['childTerm'].lower()] = row['childTerm']

print(f"Created mapping for {len(atc_lowercase_to_original)} unique ATC terms")

# Prepare lists for matching
drug_names_list = list(orig_names)
atc_names_list = list(atc_names)

def normalize(name):
    """Normalize drug names by removing salts and punctuation"""
    name = name.lower().replace('-', ' ').replace(',', '').strip()
    for salt in ['hydrochloride', 'sodium', 'potassium', 'phosphate', 'acetate', 'maleate', 'besylate', 'chloride']:
        name = name.replace(salt, '')
    return name.strip()

# Fuzzy match drug names to ATC names
matches_fuzzy = {}
for drug in drug_names_list:
    norm_drug = normalize(drug)
    match, score, _ = process.extractOne(norm_drug, [normalize(a) for a in atc_names_list], scorer=fuzz.token_sort_ratio)
    if score > 90:
        # Map back to original ATC term from atc_df
        original_atc = atc_lowercase_to_original.get(match, None)
        matches_fuzzy[drug] = original_atc
    else:
        matches_fuzzy[drug] = None

# Create mapping DataFrame
mapping_df = pd.DataFrame([
    {'drug_name': drug, 'atc_match': atc_match}
    for drug, atc_match in matches_fuzzy.items()
])

print(f"\nTotal drugs: {len(mapping_df)}")
print(f"Matched: {mapping_df['atc_match'].notna().sum()}")
print(f"Unmatched: {mapping_df['atc_match'].isna().sum()}")
print(f"\nSample matches:")
print(mapping_df[mapping_df['atc_match'].notna()].head())

Created mapping for 6146 unique ATC terms

Total drugs: 1516
Matched: 1286
Unmatched: 230

Sample matches:
       drug_name      atc_match
0     ropinirole     Ropinirole
1  carbenicillin  Carbenicillin
2   palonosetron   Palonosetron
3      naloxegol      Naloxegol
4  pivampicillin  Pivampicillin


In [136]:
# Join the ATC mapping to the filtered drug descriptions
emb_desc_filtered['drug_name_lower'] = emb_desc_filtered['Drug Name'].str.lower()
emb_desc_enriched = emb_desc_filtered.merge(
    mapping_df,
    left_on='drug_name_lower',
    right_on='drug_name',
    how='left'
)

# Display results
print(f"Total drugs in dataset: {len(emb_desc_enriched)}")
print(f"With ATC match: {emb_desc_enriched['atc_match'].notna().sum()}")
print(f"Without ATC match: {emb_desc_enriched['atc_match'].isna().sum()}\n")

# Show unmatched drugs
unmatched_drugs = emb_desc_enriched[emb_desc_enriched['atc_match'].isna()][['Drug ID', 'Drug Name', 'atc_match']]
print(f"Unmatched drugs ({len(unmatched_drugs)})")
print(unmatched_drugs)

Total drugs in dataset: 1516
With ATC match: 1286
Without ATC match: 230

Unmatched drugs (230)
      Drug ID                            Drug Name atc_match
6     DB00275                           Olmesartan      None
7     DB05259                           Glatiramer      None
20    DB11256                     Levomefolic acid      None
26    DB01016                            Glyburide      None
38    DB04897                          Lucinactant      None
...       ...                                  ...       ...
1502  DB00083               Botulinum toxin type A      None
1503  DB00017                    Salmon calcitonin      None
1504  DB00100  Coagulation Factor IX (Recombinant)      None
1508  DB00166                          Lipoic acid      None
1510  DB00263                        Sulfisoxazole      None

[230 rows x 3 columns]


/tmp/ipykernel_1019072/1180948786.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emb_desc_filtered['drug_name_lower'] = emb_desc_filtered['Drug Name'].str.lower()


In [137]:
def find_word_match(drug_name, atc_names_list):
    """Find ATC names where drug_name appears as a complete word"""
    pattern = r'\b' + re.escape(drug_name.lower()) + r'\b'
    matches = [atc for atc in atc_names_list if re.search(pattern, atc.lower())]
    return matches[0] if matches else None

# Try to recover unmatched drugs with word boundary matching
unmatched_list = emb_desc_enriched[emb_desc_enriched['atc_match'].isna()]['Drug Name'].str.lower().tolist()
word_matches = {}
for drug in unmatched_list:
    match = find_word_match(drug, atc_names_list)
    if match:
        word_matches[drug] = match

print(f"Recovered by word matching: {len(word_matches)}/{len(unmatched_list)}")
if word_matches:
    print("\nExamples:")
    for drug, atc in list(word_matches.items())[:5]:
        print(f"  {drug} -> {atc}")

# Clean up temporary column and prepare for merge
emb_desc_enriched_clean = emb_desc_enriched.drop('drug_name_lower', axis=1)

Recovered by word matching: 75/230

Examples:
  olmesartan -> olmesartan medoxomil and amlodipine
  glatiramer -> glatiramer acetate
  imipenem -> imipenem and cilastatin
  calcium chloride -> calcium chloride
  lopinavir -> lopinavir and ritonavir


In [138]:
# Join with ATC level 1 classification
# Merge on the atc_match = childTerm to get atc_class_lvl_1
emb_desc_final = emb_desc_enriched_clean.merge(
    atc_df[['childTerm', 'atc_class_lvl_1']],
    left_on='atc_match',
    right_on='childTerm',
    how='left'
).drop('childTerm', axis=1)

print("\nFinal enriched dataset with ATC level 1 classification:")
print(emb_desc_final[['Drug ID', 'Drug Name', 'atc_match', 'atc_class_lvl_1']].head(15))
print(f"\nShape: {emb_desc_final.shape}")
print(f"\nLevel 1 ATC categories in dataset:")
print(emb_desc_final['atc_class_lvl_1'].value_counts())
print(f"\nMissing ATC matches: {emb_desc_final['atc_match'].isna().sum()}")
print(f"Missing level 1 terms: {emb_desc_final['atc_class_lvl_1'].isna().sum()}")


Final enriched dataset with ATC level 1 classification:
    Drug ID     Drug Name     atc_match  \
0   DB00736  Esomeprazole  Esomeprazole   
1   DB01026  Ketoconazole  Ketoconazole   
2   DB01026  Ketoconazole  Ketoconazole   
3   DB01026  Ketoconazole  Ketoconazole   
4   DB01026  Ketoconazole  Ketoconazole   
5   DB00213  Pantoprazole  Pantoprazole   
6   DB00295      Morphine      Morphine   
7   DB00315  Zolmitriptan  Zolmitriptan   
8   DB00526   Oxaliplatin   Oxaliplatin   
9   DB00275    Olmesartan          None   
10  DB05259    Glatiramer          None   
11  DB00177     Valsartan     Valsartan   
12  DB01183      Naloxone      Naloxone   
13  DB01183      Naloxone      Naloxone   
14  DB00458    Imipramine    Imipramine   

                                      atc_class_lvl_1  
0            1: ALIMENTARY TRACT AND METABOLISM DRUGS  
1   1: SYSTEMIC HORMONAL PREPARATIONS, EXCL. SEX H...  
2                  1: ANTIINFECTIVES FOR SYSTEMIC USE  
3           1: GENITO URINARY 

In [139]:
emb_desc_final[emb_desc_final['atc_class_lvl_1'].notna()][['Drug ID', 'Drug Name', 'atc_match', 'atc_class_lvl_1']]

,Drug ID,Drug Name,atc_match,atc_class_lvl_1
0,DB00736,Esomeprazole,Esomeprazole,1: ALIMENTARY TRACT AND METABOLISM DRUGS
1,DB01026,Ketoconazole,Ketoconazole,"1: SYSTEMIC HORMONAL PREPARATIONS, EXCL. SEX H..."
2,DB01026,Ketoconazole,Ketoconazole,1: ANTIINFECTIVES FOR SYSTEMIC USE
3,DB01026,Ketoconazole,Ketoconazole,1: GENITO URINARY SYSTEM AND SEX HORMONES
4,DB01026,Ketoconazole,Ketoconazole,1: DERMATOLOGICALS
...,...,...,...,...
1881,DB00276,Amsacrine,Amsacrine,1: ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS
1882,DB00224,Indinavir,Indinavir,1: ANTIINFECTIVES FOR SYSTEMIC USE
1883,DB00229,Cefotiam,Cefotiam,1: ANTIINFECTIVES FOR SYSTEMIC USE
1884,DB00242,Cladribine,Cladribine,1: ANTINEOPLASTIC AND IMMUNOMODULATING AGENTS


# Write File

In [140]:
output_path = "/data/giobbi/embeddings/not_aligned_with_model/drug_description_enriched_atc.csv"


In [141]:
emb_desc_final.to_csv(output_path, sep="\t")
#emb_desc_masked = pd.read_csv(output_path, sep="\t", index_col=0)
emb_desc_final.head()

,Drug ID,Drug Name,Discription,Description_Masked,drug_unique_count,drug_count,pharma_class,drug_name,atc_match,atc_class_lvl_1
0,DB00736,Esomeprazole,"Esomeprazole, sold under the brand name Nexium...","<DRUG>, sold under the brand name Nexium, is a...",9,13,8,esomeprazole,Esomeprazole,1: ALIMENTARY TRACT AND METABOLISM DRUGS
1,DB01026,Ketoconazole,Ketoconazole is an imidazole antifungal agent ...,<DRUG> is an imidazole antifungal agent used i...,9,13,1,ketoconazole,Ketoconazole,"1: SYSTEMIC HORMONAL PREPARATIONS, EXCL. SEX H..."
2,DB01026,Ketoconazole,Ketoconazole is an imidazole antifungal agent ...,<DRUG> is an imidazole antifungal agent used i...,9,13,1,ketoconazole,Ketoconazole,1: ANTIINFECTIVES FOR SYSTEMIC USE
3,DB01026,Ketoconazole,Ketoconazole is an imidazole antifungal agent ...,<DRUG> is an imidazole antifungal agent used i...,9,13,1,ketoconazole,Ketoconazole,1: GENITO URINARY SYSTEM AND SEX HORMONES
4,DB01026,Ketoconazole,Ketoconazole is an imidazole antifungal agent ...,<DRUG> is an imidazole antifungal agent used i...,9,13,1,ketoconazole,Ketoconazole,1: DERMATOLOGICALS
